# OrbitTrace CV-T15 — isolated streak experiment
This notebook has been structurally checked locally; it has not been executed in Colab. Use a GPU runtime. Upload the reviewed experiment source and backend schema to Drive yourself, together with the already acquired dataset. No dataset download, public upload, or production replacement is performed. Labels mean generic streaks; ESA compact-point benchmarks are a different task. Dataset: Olivier Parisot/LIST, Zenodo 14047944, CC BY-SA 4.0; Ultralytics code/models: AGPL-3.0 or Enterprise.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import os,sys,subprocess
REPO=Path("/content/drive/MyDrive/OrbitTrace/fusion26") # edit
SOURCE=Path("/content/drive/MyDrive/OrbitTrace/StreaksYoloDataset") # edit
assert (REPO/"experiments/yolo/train.py").is_file(), "Upload the new experiment source first"
assert (REPO/"backend/app/schemas/result.py").is_file()
assert SOURCE.is_dir()
os.chdir(REPO)
os.environ["PYTHONPATH"]=str(REPO)+":"+str(REPO/"backend")
os.environ["CV_T15_CACHE"]="/content/cv_t15_cache"
OUT=Path("/content/cv_t15_cache");OUT.mkdir(exist_ok=True)
def run(*args):subprocess.run([sys.executable,*map(str,args)],check=True)


In [ ]:
import torch
print(sys.version,torch.__version__,torch.version.cuda,torch.cuda.is_available())
assert torch.cuda.is_available(), "Select a GPU runtime before training"
print(torch.cuda.get_device_name(0))
run("-m","pip","install","-r",REPO/"experiments/yolo/requirements.txt")
run("-m","pip","check")


Re-audit the Drive dataset, then create a local training view. Six zero-size labels were seen in the Windows audit; invalid samples must be explicitly excluded, not changed. The scripts preserve official splits and conservatively exclude exact/cross-split pHash matches. No capture-session metadata exists; this does not certify session independence. Missing labels are assumed negatives under YOLO conventions.

In [ ]:
run("-m","experiments.yolo.audit","--source",SOURCE,"--output",OUT/"audit")


The audit exits with an error when malformed samples exist, while saving its report. Inspect that report. If errors consist of the known zero-size labels, run the explicit exclusion step below. Stop for any new corruption or unexplained errors.

In [ ]:
import json
audit=json.loads((OUT/"audit/dataset_audit.json").read_text())
print(audit["counts"],audit["errors"],audit["cross_split_near_pairs"])
run("-m","experiments.yolo.prepare","--audit",OUT/"audit/dataset_audit.json","--output",OUT/"dataset","--exclude-invalid")
run("-m","experiments.yolo.fetch_weights","--output",OUT/"weights/yolo26n.pt")


In [ ]:
run("-m","experiments.yolo.train","--data",OUT/"dataset/smoke.yaml","--weights",OUT/"weights/yolo26n.pt","--output",OUT/"smoke","--epochs",1,"--imgsz",320,"--batch",4,"--device",0)
smoke=json.loads((OUT/"smoke/training_evidence.json").read_text())
assert smoke["state"]=="completed" and smoke["changed_parameter_tensors"]>0
print(smoke)


Only proceed when the smoke run completes with finite losses, actual parameter updates, best/last checkpoints and validation. The main run starts from pretrained weights rather than selecting settings on test results. Validation chooses best.pt; test is evaluated once after the training decision.

In [ ]:
run("-m","experiments.yolo.train","--data",OUT/"dataset/dataset.yaml","--weights",OUT/"weights/yolo26n.pt","--output",OUT/"train","--epochs",10,"--imgsz",640,"--batch",4,"--device",0,"--time-hours",0.25)
run("-m","experiments.yolo.evaluate","--data",OUT/"dataset/dataset.yaml","--weights",OUT/"train/run/weights/best.pt","--output",OUT/"test","--split","test","--device",0,"--conf",0.25)


In [ ]:
from IPython.display import display,Image
report=json.loads((OUT/"test/evaluation.json").read_text())
print({k:v for k,v in report.items() if k not in ("by_image","failure_cases")})
display(Image(filename=str(OUT/"test/failure_examples.png")))
# Copy only outputs you want to retain back to your Drive; Colab storage is ephemeral.
import shutil
SAVE=REPO/"artifacts/reports/cv_t15_colab"
assert not SAVE.exists()
shutil.copytree(OUT/"train",SAVE/"train")
shutil.copytree(OUT/"test",SAVE/"test")
